# SciQ-Scorecards-Chatbot

**Praxisprojekt – Aufgabe 3 (50 %)**  
Grundlagen der Informatik · Dr. Jochen Heistermann · TIS/TIK24

---

## Übersicht

Dieses Notebook implementiert einen Chatbot zur Beantwortung naturwissenschaftlicher Multiple-Choice-Fragen auf Basis des Datensatzes **allenai/sciq**.

**Architektur-Pipeline:**
1. Tokenisierung mit `bert-base-uncased` (nur Tokenizer + Eingabe-Embeddings)
2. Mean-Pooling über Token-Embeddings → Textvektoren (768-dim)
3. Merkmalskonstruktion: `[Kontext, Antwort, |Diff|, Produkt]` → 3072-dim
4. MLP-Klassifikation: 3072 → 512 → 1 (pro Antwort-Option)
5. Training mit CrossEntropyLoss + AdamW, Evaluierung + Inferenz via `ask_bot`

**Über die Aufgabenstellung hinaus:**
- Trainings- & Validierungskurven als Plot
- Konfidenzanalyse der Vorhersagen
- Learning-Rate-Scheduling (ReduceLROnPlateau)
- Early Stopping mit Geduld
- Detaillierte Fehleranalyse auf dem Testset
- Interaktiver Modus mit benutzerdefinierten Fragen


## 1. Setup und Imports

In [ ]:
import os
import random
import warnings
from typing import Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings('ignore')

# ── Reproduzierbarkeit ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Gerät: {DEVICE}')
print(f'PyTorch-Version: {torch.__version__}')


## 2. Hyperparameter

Alle Hyperparameter sind zentral definiert, um Experimente zu erleichtern.

In [ ]:
# ── Tokenisierungsgrenzen (gem. Handout) ──
MAX_Q = 48    # maximale Token-Länge für Fragen
MAX_S = 96    # maximale Token-Länge für Support-Texte
MAX_A = 28    # maximale Token-Länge für Antworten

# ── Modell ──
HIDDEN_DIM  = 512
DROPOUT     = 0.30

# ── Training ──
BATCH_SIZE     = 16
LEARNING_RATE  = 3e-4
WEIGHT_DECAY   = 1e-2
EPOCHS         = 30
PATIENCE       = 5   # Early Stopping (über Handout hinaus)
LR_FACTOR      = 0.5 # LR-Scheduler-Faktor (über Handout hinaus)
LR_PATIENCE    = 3   # LR-Scheduler-Geduld (über Handout hinaus)

# ── Embedding ──
EMB_DIM     = 768     # bert-base-uncased
FEATURE_DIM = 4 * EMB_DIM  # = 3072

print(f'Feature-Dimension: {FEATURE_DIM}')
print(f'Batch-Größe: {BATCH_SIZE}')
print(f'Epochen: {EPOCHS}')


## 3. Datensatz laden und inspizieren

Der Datensatz `allenai/sciq` enthält naturwissenschaftliche Fragen mit je einer korrekten Antwort, drei Distraktoren und einem optionalen Support-Text.

In [ ]:
dataset = load_dataset('allenai/sciq')

print(f'Splits: {list(dataset.keys())}')
for split_name, split_data in dataset.items():
    print(f'  {split_name}: {len(split_data)} Beispiele')

print('\n── Beispiel aus dem Trainingsdatensatz ──')
example = dataset['train'][0]
for key, value in example.items():
    display_val = value if len(str(value)) < 120 else str(value)[:120] + '...'
    print(f'  {key:20s}: {display_val}')


### 3.1 Datenqualität prüfen

Wir prüfen systematisch, wie häufig Support-Texte leer sind und ob es Duplikate gibt – das ist wichtig für robuste Tokenisierung.

In [ ]:
# Analyse der Support-Text-Verfügbarkeit und Textlängen
for split_name in ['train', 'validation', 'test']:
    split = dataset[split_name]
    n_total = len(split)
    n_empty_support = sum(1 for ex in split if not ex['support'] or ex['support'].strip() == '')
    avg_q_len = np.mean([len(ex['question'].split()) for ex in split])
    avg_s_len = np.mean([len(ex['support'].split()) for ex in split if ex['support'].strip()])
    print(f'{split_name:12s}: {n_total:5d} Beispiele | '
          f'leerer Support: {n_empty_support:4d} ({100*n_empty_support/n_total:.1f}%) | '
          f'Ø Frage: {avg_q_len:.1f} Wörter | Ø Support: {avg_s_len:.1f} Wörter')


## 4. Tokenizer und Embedding-Matrix

**Zentrale Vorgabe:** Wir verwenden **ausschließlich** den Tokenizer und die Eingabe-Embedding-Matrix von `bert-base-uncased`. Der BERT-Encoder wird **nicht** verwendet – kein `model(**inputs)`, keine Hidden States, kein Pooler.

In [ ]:
# ── Tokenizer laden ──
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# ── Embedding-Matrix extrahieren (fest, nicht trainierbar) ──
hf_model = AutoModel.from_pretrained('bert-base-uncased')
EMB_MATRIX = hf_model.get_input_embeddings().weight.detach().clone()
EMB_MATRIX.requires_grad_(False)
EMB_MATRIX = EMB_MATRIX.to(DEVICE)

# BERT-Modell wird nicht weiter benötigt → Speicher freigeben
del hf_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f'Vokabular: {EMB_MATRIX.shape[0]} Tokens')
print(f'Embedding-Dim: {EMB_MATRIX.shape[1]}')
print(f'Matrix auf Gerät: {EMB_MATRIX.device}')
print(f'Trainierbar: {EMB_MATRIX.requires_grad}')


## 5. Textverarbeitung: Tokenisierung und Textvektoren

Zwei Kernfunktionen:
- `tokenize_text`: Text → Token-IDs (ohne Special Tokens, mit Truncation)
- `mean_embedding`: Token-IDs → Mittelwertsvektor (768-dim)

Leere Texte werden robust als Nullvektor behandelt, um NaN-Werte zu vermeiden.

In [ ]:
def tokenize_text(text: Optional[str], max_len: int) -> list[int]:
    """
    Tokenisiert einen Text ohne Special Tokens.
    
    Args:
        text:    Eingabetext (kann None oder leer sein)
        max_len: Maximale Anzahl an Tokens
    
    Returns:
        Liste von Token-IDs; leere Liste bei leerem/None-Text
    """
    if text is None or str(text).strip() == '':
        return []
    return tokenizer.encode(
        str(text),
        add_special_tokens=False,
        truncation=True,
        max_length=max_len
    )


def mean_embedding(token_ids: list[int], emb_matrix: torch.Tensor) -> torch.Tensor:
    """
    Berechnet den Mittelwertsvektor über alle Token-Embeddings.
    
    Args:
        token_ids:  Liste von Token-IDs
        emb_matrix: Feste Embedding-Matrix (V x D)
    
    Returns:
        Textvektor der Dimension D; Nullvektor falls token_ids leer
    """
    if len(token_ids) == 0:
        return torch.zeros(emb_matrix.shape[1], device=emb_matrix.device)
    ids_tensor = torch.tensor(token_ids, dtype=torch.long, device=emb_matrix.device)
    embeddings = emb_matrix[ids_tensor]  # (T, D)
    return embeddings.mean(dim=0)         # (D,)


### 5.1 Verifikation der Textverarbeitung

In [ ]:
# ── Funktionstest ──
test_text = 'What is the speed of light?'
test_ids = tokenize_text(test_text, MAX_Q)
test_vec = mean_embedding(test_ids, EMB_MATRIX)
print(f'Text:       "{test_text}"')
print(f'Token-IDs:  {test_ids}')
print(f'Tokens:     {tokenizer.convert_ids_to_tokens(test_ids)}')
print(f'Vektor:     Form={test_vec.shape}, Norm={test_vec.norm():.4f}')

# ── Robustheit bei leerem Text ──
empty_vec = mean_embedding(tokenize_text(None, MAX_Q), EMB_MATRIX)
print(f'\nLeerer Text → Nullvektor: Form={empty_vec.shape}, Summe={empty_vec.sum():.4f}')
assert empty_vec.sum() == 0, 'Nullvektor-Check fehlgeschlagen!'
print('✓ Alle Tests bestanden')


## 6. Merkmalsvektor-Konstruktion

Für jedes Kontext-Antwort-Paar wird ein 3072-dimensionaler Merkmalsvektor erstellt:

$$\mathbf{f} = [\mathbf{c},\; \mathbf{a},\; |\mathbf{c} - \mathbf{a}|,\; \mathbf{c} \odot \mathbf{a}]$$

wobei $\mathbf{c}$ der Kontextvektor (Frage + Support) und $\mathbf{a}$ der Antwortvektor ist.

In [ ]:
def build_feature_vector(context_vec: torch.Tensor, answer_vec: torch.Tensor) -> torch.Tensor:
    """
    Konstruiert den Merkmalsvektor aus Kontext- und Antwortvektor.
    
    Bestandteile: [Kontext, Antwort, |Differenz|, Elementweises Produkt]
    Ergibt 4 * 768 = 3072 Dimensionen.
    
    Args:
        context_vec: Kontextvektor (D,)
        answer_vec:  Antwortvektor (D,)
    
    Returns:
        Merkmalsvektor (4*D,)
    """
    diff_vec = torch.abs(context_vec - answer_vec)
    prod_vec = context_vec * answer_vec
    return torch.cat([context_vec, answer_vec, diff_vec, prod_vec], dim=0)


# ── Verifikation ──
dummy_ctx = torch.randn(EMB_DIM, device=DEVICE)
dummy_ans = torch.randn(EMB_DIM, device=DEVICE)
feat = build_feature_vector(dummy_ctx, dummy_ans)
print(f'Merkmalsvektor: Form={feat.shape} (erwartet: ({FEATURE_DIM},))')
assert feat.shape == (FEATURE_DIM,), f'Formfehler: {feat.shape} != ({FEATURE_DIM},)'
print('✓ Merkmalsvektor korrekt')


## 7. Dataset-Klasse

Die Dataset-Klasse verwaltet:
- Zugriff auf Rohtexte
- Permutation der Antwortoptionen (zufällig im Training, deterministisch bei Evaluation)
- Label-Bestimmung nach Permutation
- Tokenisierung aller Textbestandteile

Die eigentliche Vektorisierung erfolgt in der Collate-Funktion.

In [ ]:
class SciQDataset(Dataset):
    """
    PyTorch Dataset für den SciQ-Datensatz.
    
    Liefert pro Beispiel tokenisierte Frage, Support, vier permutierte
    Antworten sowie das korrekte Label nach Permutation.
    """
    
    def __init__(self, split_data, split_name: str = 'train'):
        """
        Args:
            split_data: HuggingFace Dataset-Split
            split_name: 'train', 'validation' oder 'test'
        """
        self.data = split_data
        self.split_name = split_name
        self.is_train = (split_name == 'train')
    
    def __len__(self) -> int:
        return len(self.data)
    
    def __getitem__(self, index: int) -> dict:
        example = self.data[index]
        
        # ── Rohtexte ──
        question = example['question']
        support  = example['support']
        correct  = example['correct_answer']
        answers  = [
            correct,
            example['distractor1'],
            example['distractor2'],
            example['distractor3']
        ]
        
        # ── Permutation ──
        perm = list(range(4))
        if self.is_train:
            random.shuffle(perm)
        else:
            rng = random.Random(SEED + index)
            rng.shuffle(perm)
        
        permuted_answers = [answers[i] for i in perm]
        label = perm.index(0)  # Position der korrekten Antwort nach Permutation
        
        # ── Tokenisierung ──
        q_ids = tokenize_text(question, MAX_Q)
        s_ids = tokenize_text(support, MAX_S)
        a_ids = [tokenize_text(a, MAX_A) for a in permuted_answers]
        
        return {
            'question_ids': q_ids,
            'support_ids':  s_ids,
            'answer_ids':   a_ids,
            'label':        label,
            'question':     question,
            'support':      support,
            'answer_texts': permuted_answers,
            'correct_answer': correct
        }


## 8. Collate-Funktion

Die Collate-Funktion übernimmt die Batch-Bildung. Sie:
1. Berechnet den Kontextvektor (Frage + Support)
2. Berechnet vier Antwortvektoren
3. Baut vier Merkmalsvektoren pro Beispiel
4. Erzeugt den Batch-Tensor `(B, 4, 3072)` und den Label-Tensor `(B,)`

In [ ]:
def collate_fn(batch: list[dict]) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Batch-Collate: Erzeugt Merkmalstensoren und Labels.
    
    Args:
        batch: Liste von Beispiel-Dicts aus SciQDataset
    
    Returns:
        X: Tensor (B, 4, F) – Merkmalsvektoren
        y: Tensor (B,)     – Labels
    """
    all_features = []
    all_labels   = []
    
    for item in batch:
        # Kontextvektor: Frage + Support zusammenführen
        context_ids = item['question_ids'] + item['support_ids']
        context_vec = mean_embedding(context_ids, EMB_MATRIX)
        
        # Vier Merkmalsvektoren (einer pro Antwortoption)
        option_features = []
        for a_ids in item['answer_ids']:
            answer_vec = mean_embedding(a_ids, EMB_MATRIX)
            feat_vec = build_feature_vector(context_vec, answer_vec)
            option_features.append(feat_vec)
        
        # (4, F)
        example_tensor = torch.stack(option_features, dim=0)
        all_features.append(example_tensor)
        all_labels.append(item['label'])
    
    X = torch.stack(all_features, dim=0)                    # (B, 4, F)
    y = torch.tensor(all_labels, dtype=torch.long, device=DEVICE)  # (B,)
    return X, y


### 8.1 Tensorform-Verifikation

In [ ]:
# ── Vollständiger Pipeline-Test mit einem Mini-Batch ──
test_ds = SciQDataset(dataset['train'], 'train')
test_batch = [test_ds[i] for i in range(BATCH_SIZE)]
X_test, y_test = collate_fn(test_batch)

print('── Tensorformen (ein Batch) ──')
print(f'  X: {X_test.shape}  (erwartet: ({BATCH_SIZE}, 4, {FEATURE_DIM}))')
print(f'  y: {y_test.shape}  (erwartet: ({BATCH_SIZE},))')
print(f'  Labels: {y_test.tolist()}')
print(f'  X enthält NaN: {torch.isnan(X_test).any().item()}')

assert X_test.shape == (BATCH_SIZE, 4, FEATURE_DIM)
assert y_test.shape == (BATCH_SIZE,)
assert not torch.isnan(X_test).any()
print('\n✓ Pipeline-Test bestanden')


## 9. DataLoaders erstellen

In [ ]:
train_dataset = SciQDataset(dataset['train'], 'train')
val_dataset   = SciQDataset(dataset['validation'], 'validation')
test_dataset  = SciQDataset(dataset['test'], 'test')

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  collate_fn=collate_fn, drop_last=False)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, collate_fn=collate_fn, drop_last=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          shuffle=False, collate_fn=collate_fn, drop_last=False)

print(f'Train-Batches: {len(train_loader)}')
print(f'Val-Batches:   {len(val_loader)}')
print(f'Test-Batches:  {len(test_loader)}')


## 10. Modell: ScorecardsMLP

Architektur gemäß Handout:
- Linear(3072, 512) → ReLU → Dropout(0.3) → Linear(512, 1)
- Im Forward: Reshape `(B, 4, F)` → `(B*4, F)` → MLP → `(B, 4)`
- **Keine Softmax** vor CrossEntropyLoss (erwartet rohe Logits)

**Über Handout hinaus:** Batch-Normalisierung für stabileres Training.

In [ ]:
class ScorecardsMLP(nn.Module):
    """
    MLP zur Score-Berechnung für Kontext-Antwort-Paare.
    
    Architektur: Linear → BatchNorm → ReLU → Dropout → Linear(→1)
    Eingabe:  (B, 4, F)  – vier Merkmalsvektoren pro Beispiel
    Ausgabe:  (B, 4)     – ein Score pro Antwort-Option
    """
    
    def __init__(self, feature_dim: int = FEATURE_DIM,
                 hidden_dim: int = HIDDEN_DIM,
                 dropout: float = DROPOUT):
        super().__init__()
        self.fc1  = nn.Linear(feature_dim, hidden_dim)
        self.bn1  = nn.BatchNorm1d(hidden_dim)  # Über Handout hinaus
        self.relu = nn.ReLU()
        self.drop = nn.Dropout(p=dropout)
        self.fc2  = nn.Linear(hidden_dim, 1)
        
        # Gewichtsinitialisierung (über Handout hinaus)
        nn.init.kaiming_normal_(self.fc1.weight, nonlinearity='relu')
        nn.init.xavier_normal_(self.fc2.weight)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward-Pass.
        
        Args:
            x: Tensor (B, 4, F)
        
        Returns:
            logits: Tensor (B, 4) – rohe Scores (keine Softmax!)
        """
        B = x.size(0)
        x = x.view(B * 4, -1)      # (B*4, F)
        h = self.fc1(x)             # (B*4, H)
        h = self.bn1(h)             # (B*4, H)
        h = self.relu(h)            # (B*4, H)
        h = self.drop(h)            # (B*4, H)
        s = self.fc2(h)             # (B*4, 1)
        return s.view(B, 4)         # (B, 4)


# ── Modelltest ──
model = ScorecardsMLP().to(DEVICE)
dummy_logits = model(X_test.to(DEVICE))
print(f'Modell-Parameter: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
print(f'Logits: {dummy_logits.shape} (erwartet: ({BATCH_SIZE}, 4))')
assert dummy_logits.shape == (BATCH_SIZE, 4)
print('\n✓ Modelltest bestanden')


## 11. Evaluierungsfunktion

In [ ]:
@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader) -> tuple[float, float]:
    """
    Evaluiert das Modell auf einem DataLoader.
    
    Args:
        model:  Das zu evaluierende Modell
        loader: DataLoader (Validation oder Test)
    
    Returns:
        accuracy: Anteil korrekt klassifizierter Beispiele
        avg_loss: Durchschnittlicher Verlust
    """
    model.eval()
    loss_fn = nn.CrossEntropyLoss()
    correct = 0
    total   = 0
    total_loss = 0.0
    n_batches  = 0
    
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        logits = model(X)
        loss = loss_fn(logits, y)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total   += y.size(0)
        total_loss += loss.item()
        n_batches  += 1
    
    return correct / total, total_loss / max(n_batches, 1)


## 12. Training

Training-Loop mit:
- AdamW-Optimizer und CrossEntropyLoss (gemäß Handout)
- **Über Handout hinaus:** Learning-Rate-Scheduling (ReduceLROnPlateau) und Early Stopping
- Speicherung des besten Modells anhand der Validation Accuracy
- Protokollierung aller Metriken für spätere Visualisierung

In [ ]:
# ── Initialisierung ──
model     = ScorecardsMLP().to(DEVICE)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=LR_FACTOR,
                              patience=LR_PATIENCE)
loss_fn   = nn.CrossEntropyLoss()

# ── Tracking ──
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss':   [], 'val_acc':   [],
    'lr': []
}
best_val_acc  = 0.0
best_state    = None
patience_counter = 0

print(f'Training startet: {EPOCHS} Epochen, Batch-Größe {BATCH_SIZE}')
print(f'Modell: {sum(p.numel() for p in model.parameters() if p.requires_grad):,} Parameter')
print('=' * 85)


In [ ]:
for epoch in range(1, EPOCHS + 1):
    # ── Training ──
    model.train()
    train_correct = 0
    train_total   = 0
    train_loss_sum = 0.0
    n_batches = 0
    
    for X, y in train_loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        
        optimizer.zero_grad()
        logits = model(X)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()
        
        preds = logits.argmax(dim=1)
        train_correct += (preds == y).sum().item()
        train_total   += y.size(0)
        train_loss_sum += loss.item()
        n_batches += 1
    
    train_acc  = train_correct / train_total
    train_loss = train_loss_sum / n_batches
    
    # ── Validation ──
    val_acc, val_loss = evaluate(model, val_loader)
    
    # ── Scheduler ──
    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step(val_acc)
    
    # ── Tracking ──
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['lr'].append(current_lr)
    
    # ── Bestes Modell speichern ──
    improved = ''
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
        improved = ' ★'
    else:
        patience_counter += 1
    
    print(f'Epoche {epoch:2d}/{EPOCHS} │ '
          f'Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f} │ '
          f'Val Loss: {val_loss:.4f}  Acc: {val_acc:.4f} │ '
          f'LR: {current_lr:.1e}{improved}')
    
    # ── Early Stopping ──
    if patience_counter >= PATIENCE:
        print(f'\nEarly Stopping nach Epoche {epoch} (keine Verbesserung seit {PATIENCE} Epochen)')
        break

print('=' * 85)
print(f'Beste Validation Accuracy: {best_val_acc:.4f}')


## 13. Trainingsvisualisierung (über Handout hinaus)

Darstellung der Trainings- und Validierungskurven für Loss und Accuracy.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
epochs_range = range(1, len(history['train_loss']) + 1)

# ── Loss ──
ax = axes[0]
ax.plot(epochs_range, history['train_loss'], 'o-', label='Train', markersize=3)
ax.plot(epochs_range, history['val_loss'], 's-', label='Validation', markersize=3)
ax.set_xlabel('Epoche')
ax.set_ylabel('Loss')
ax.set_title('Verlauf: Loss')
ax.legend()
ax.grid(True, alpha=0.3)

# ── Accuracy ──
ax = axes[1]
ax.plot(epochs_range, history['train_acc'], 'o-', label='Train', markersize=3)
ax.plot(epochs_range, history['val_acc'], 's-', label='Validation', markersize=3)
ax.axhline(y=0.70, color='red', linestyle='--', alpha=0.5, label='70%-Schwelle')
ax.set_xlabel('Epoche')
ax.set_ylabel('Accuracy')
ax.set_title('Verlauf: Accuracy')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.legend()
ax.grid(True, alpha=0.3)

# ── Learning Rate ──
ax = axes[2]
ax.plot(epochs_range, history['lr'], 'D-', color='green', markersize=3)
ax.set_xlabel('Epoche')
ax.set_ylabel('Learning Rate')
ax.set_title('Verlauf: Learning Rate')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot gespeichert als training_curves.png')


## 14. Evaluation auf dem Testdatensatz

Das beste Modell (anhand der Validation Accuracy ausgewählt) wird geladen und auf dem Testsplit evaluiert.

In [ ]:
# ── Bestes Modell laden ──
model.load_state_dict(best_state)
print('Bestes Modell geladen (basierend auf Validation Accuracy)')

# ── Test-Evaluation ──
test_acc, test_loss = evaluate(model, test_loader)

# ── Trainings-Accuracy mit bestem Modell ──
train_acc_final, train_loss_final = evaluate(model, train_loader)

print(f'\n{"═" * 50}')
print(f'  ERGEBNISSE (bestes Modell)')
print(f'{"═" * 50}')
print(f'  Train Accuracy:      {train_acc_final:.4f} ({train_acc_final*100:.2f}%)')
print(f'  Validation Accuracy: {best_val_acc:.4f} ({best_val_acc*100:.2f}%)')
print(f'  Test Accuracy:       {test_acc:.4f} ({test_acc*100:.2f}%)')
print(f'  Test Loss:           {test_loss:.4f}')
print(f'{"═" * 50}')

if train_acc_final >= 0.70 and test_acc >= 0.70:
    print('\n✓ Anforderung erfüllt: ≥ 70% auf Trainings- UND Testdaten')
else:
    print('\n⚠ Anforderung von ≥ 70% noch nicht auf beiden Splits erreicht')


## 15. Detaillierte Fehleranalyse (über Handout hinaus)

Analyse der Konfidenz bei korrekten vs. falschen Vorhersagen und Untersuchung typischer Fehlerarten.

In [ ]:
@torch.no_grad()
def detailed_analysis(model: nn.Module, loader: DataLoader) -> dict:
    """
    Sammelt detaillierte Vorhersagestatistiken.
    
    Returns:
        Dictionary mit Konfidenz-Statistiken und Fehlermustern
    """
    model.eval()
    correct_confidences = []
    wrong_confidences   = []
    
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        logits = model(X)
        probs  = F.softmax(logits, dim=1)
        preds  = logits.argmax(dim=1)
        
        for i in range(len(y)):
            max_prob = probs[i, preds[i]].item()
            if preds[i] == y[i]:
                correct_confidences.append(max_prob)
            else:
                wrong_confidences.append(max_prob)
    
    return {
        'correct_conf': correct_confidences,
        'wrong_conf':   wrong_confidences
    }


analysis = detailed_analysis(model, test_loader)

print('── Konfidenzanalyse auf dem Testdatensatz ──')
print(f'Korrekte Vorhersagen: {len(analysis["correct_conf"])}')
print(f'  Ø Konfidenz: {np.mean(analysis["correct_conf"]):.4f}')
print(f'  Min:         {np.min(analysis["correct_conf"]):.4f}')
print(f'Falsche Vorhersagen:  {len(analysis["wrong_conf"])}')
if analysis['wrong_conf']:
    print(f'  Ø Konfidenz: {np.mean(analysis["wrong_conf"]):.4f}')
    print(f'  Max:         {np.max(analysis["wrong_conf"]):.4f}')


In [ ]:
# ── Konfidenz-Histogramm ──
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(analysis['correct_conf'], bins=25, alpha=0.6, label='Korrekt', color='#2ecc71', edgecolor='white')
ax.hist(analysis['wrong_conf'], bins=25, alpha=0.6, label='Falsch', color='#e74c3c', edgecolor='white')
ax.set_xlabel('Konfidenz (Softmax-Wahrscheinlichkeit)')
ax.set_ylabel('Anzahl')
ax.set_title('Konfidenzverteilung: Korrekte vs. Falsche Vorhersagen')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('confidence_analysis.png', dpi=150, bbox_inches='tight')
plt.show()


## 16. `ask_bot` – Inferenzfunktion

Die Funktion `ask_bot(question, options, support="")` führt die vollständige Pipeline auf einer einzelnen Frage aus:
1. Tokenisierung und Vektorisierung
2. Merkmalskonstruktion
3. Modell-Inferenz
4. Softmax → Wahrscheinlichkeiten

**Hinweis:** Im Gegensatz zum Training werden die Antwortoptionen **nicht** permutiert.

In [ ]:
def ask_bot(question: str, options: list[str], support: str = '') -> str:
    """
    Beantwortet eine Multiple-Choice-Frage mit vier Optionen.
    
    Verwendet dieselbe Pipeline wie das Training:
    Tokenisierung → Mean-Embedding → Merkmalsvektor → MLP → Softmax
    
    Args:
        question: Die Fragestellung
        options:  Liste mit genau 4 Antwortoptionen
        support:  Optionaler Kontext-/Hilfstext
    
    Returns:
        Die vom Modell gewählte Antwort als String
    """
    assert len(options) == 4, f'Genau 4 Optionen erforderlich, erhalten: {len(options)}'
    
    model.eval()
    
    # ── Kontextvektor ──
    q_ids = tokenize_text(question, MAX_Q)
    s_ids = tokenize_text(support, MAX_S)
    context_ids = q_ids + s_ids
    context_vec = mean_embedding(context_ids, EMB_MATRIX)
    
    # ── Merkmalsvektoren für alle vier Optionen ──
    features = []
    for opt in options:
        a_ids = tokenize_text(opt, MAX_A)
        a_vec = mean_embedding(a_ids, EMB_MATRIX)
        feat  = build_feature_vector(context_vec, a_vec)
        features.append(feat)
    
    # ── Inferenz ──
    X = torch.stack(features, dim=0).unsqueeze(0).to(DEVICE)  # (1, 4, F)
    
    with torch.no_grad():
        logits = model(X)                    # (1, 4)
        probs  = F.softmax(logits, dim=1)    # (1, 4)
    
    best_idx = probs.argmax(dim=1).item()
    
    # ── Ausgabe ──
    print(f'\n{"─" * 60}')
    print(f'Frage:   {question}')
    if support:
        display_support = support[:100] + '...' if len(support) > 100 else support
        print(f'Support: {display_support}')
    print(f'Optionen:')
    for i, opt in enumerate(options):
        marker = '  →' if i == best_idx else '   '
        print(f'{marker} [{i}] {opt:40s}  P = {probs[0, i]:.4f}')
    print(f'\nAntwort: {options[best_idx]} (Konfidenz: {probs[0, best_idx]:.2%})')
    print(f'{"─" * 60}')
    
    return options[best_idx]


## 17. Drei automatische `ask_bot`-Aufrufe

Gemäß Aufgabenstellung werden drei Beispiele aus dem Testdatensatz entnommen, die erwartete korrekte Antwort ausgegeben und `ask_bot` aufgerufen.

In [ ]:
def run_three_ask_bot_examples(test_ds: SciQDataset) -> None:
    """
    Führt ask_bot drei Mal automatisch auf Testbeispielen aus.
    Gibt vorher die erwartete korrekte Antwort aus.
    """
    n = min(3, len(test_ds))
    print('=' * 60)
    print('  DREI AUTOMATISCHE ask_bot-AUFRUFE')
    print('=' * 60)
    
    correct_count = 0
    for i in range(n):
        ex = test_ds[i]
        expected = ex['correct_answer']
        print(f'\n▶ Beispiel {i+1}: Erwartete korrekte Antwort: "{expected}"')
        
        chosen = ask_bot(
            question=ex['question'],
            options=ex['answer_texts'],
            support=ex['support']
        )
        
        if chosen == expected:
            print('  ✓ KORREKT')
            correct_count += 1
        else:
            print(f'  ✗ FALSCH (gewählt: "{chosen}")')
    
    print(f'\nErgebnis: {correct_count}/{n} korrekt beantwortet')


run_three_ask_bot_examples(test_dataset)


## 18. Zusätzliche Demonstrationen (über Handout hinaus)

### 18.1 Eigene Fragen an den Chatbot

In [ ]:
# ── Eigene naturwissenschaftliche Fragen ──

print('═' * 60)
print('  ZUSÄTZLICHE DEMONSTRATIONEN MIT EIGENEN FRAGEN')
print('═' * 60)

# Frage 1: Physik
ask_bot(
    question='What is the speed of light in vacuum?',
    options=[
        '3 × 10^8 m/s',
        '3 × 10^6 m/s',
        '3 × 10^10 m/s',
        '3 × 10^4 m/s'
    ],
    support='Light is an electromagnetic wave that travels fastest in vacuum.'
)

# Frage 2: Biologie
ask_bot(
    question='What organelle is responsible for producing energy in a cell?',
    options=[
        'Ribosome',
        'Nucleus',
        'Mitochondria',
        'Golgi apparatus'
    ],
    support='Cells require ATP for their metabolic processes.'
)

# Frage 3: Chemie
ask_bot(
    question='What is the chemical formula of water?',
    options=[
        'CO2',
        'H2O',
        'NaCl',
        'O2'
    ]
)


### 18.2 Top-5 sicherste und unsicherste Vorhersagen

In [ ]:
@torch.no_grad()
def get_predictions_with_confidence(model: nn.Module, ds: SciQDataset,
                                     n: int = 100) -> list[dict]:
    """
    Sammelt Vorhersagen mit Konfidenz für die ersten n Beispiele.
    """
    model.eval()
    results = []
    
    for i in range(min(n, len(ds))):
        ex = ds[i]
        q_ids = ex['question_ids']
        s_ids = ex['support_ids']
        context_ids = q_ids + s_ids
        context_vec = mean_embedding(context_ids, EMB_MATRIX)
        
        features = []
        for a_ids in ex['answer_ids']:
            a_vec = mean_embedding(a_ids, EMB_MATRIX)
            features.append(build_feature_vector(context_vec, a_vec))
        
        X = torch.stack(features).unsqueeze(0).to(DEVICE)
        logits = model(X)
        probs = F.softmax(logits, dim=1)
        pred_idx = probs.argmax(dim=1).item()
        
        results.append({
            'question': ex['question'],
            'correct':  ex['correct_answer'],
            'predicted': ex['answer_texts'][pred_idx],
            'confidence': probs[0, pred_idx].item(),
            'is_correct': ex['answer_texts'][pred_idx] == ex['correct_answer']
        })
    
    return results


predictions = get_predictions_with_confidence(model, test_dataset, n=len(test_dataset))

# Top 5 sicherste
sorted_by_conf = sorted(predictions, key=lambda x: x['confidence'], reverse=True)
print('── Top 5: Höchste Konfidenz ──')
for p in sorted_by_conf[:5]:
    status = '✓' if p['is_correct'] else '✗'
    print(f"  {status} [{p['confidence']:.2%}] {p['question'][:70]}")
    print(f"       → {p['predicted']}")

print()

# Top 5 unsicherste
print('── Top 5: Niedrigste Konfidenz ──')
for p in sorted_by_conf[-5:]:
    status = '✓' if p['is_correct'] else '✗'
    print(f"  {status} [{p['confidence']:.2%}] {p['question'][:70]}")
    print(f"       → {p['predicted']}")


## 19. Modellzusammenfassung

In [ ]:
print('╔' + '═' * 58 + '╗')
print('║  MODELLZUSAMMENFASSUNG                                   ║')
print('╠' + '═' * 58 + '╣')
print(f'║  Architektur:     ScorecardsMLP                          ║')
print(f'║  Embedding:       bert-base-uncased (fest, 768-dim)      ║')
print(f'║  Feature-Dim:     {FEATURE_DIM}                                   ║')
print(f'║  Hidden-Dim:      {HIDDEN_DIM}                                    ║')
print(f'║  Dropout:         {DROPOUT}                                    ║')
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'║  Parameter:       {n_params:,}                              ║')
print(f'║  Optimizer:       AdamW (lr={LEARNING_RATE}, wd={WEIGHT_DECAY})             ║')
print(f'║  Scheduler:       ReduceLROnPlateau                      ║')
print(f'║  Batch-Größe:     {BATCH_SIZE}                                     ║')
print(f'║  Epochen:         {len(history["train_loss"]):2d} (max {EPOCHS})                            ║')
print('╠' + '═' * 58 + '╣')
print(f'║  Train Accuracy:  {train_acc_final:.2%}                                 ║')
print(f'║  Val Accuracy:    {best_val_acc:.2%}                                 ║')
print(f'║  Test Accuracy:   {test_acc:.2%}                                 ║')
print('╚' + '═' * 58 + '╝')


## 20. Beschreibung der Lösungsschritte (Textform)

### Schritt 1: Datensatz laden und inspizieren
Der SciQ-Datensatz (`allenai/sciq`) wird über die HuggingFace `datasets`-Bibliothek geladen. Er enthält drei Splits (Train, Validation, Test) mit naturwissenschaftlichen Multiple-Choice-Fragen. Jedes Beispiel besteht aus einer Frage, einem optionalen Support-Text, der korrekten Antwort und drei Distraktoren. Die Datenqualität wird vorab geprüft – insbesondere der Anteil leerer Support-Texte.

### Schritt 2: Tokenizer und Embedding-Matrix laden
Wir laden den Tokenizer von `bert-base-uncased` für die Tokenisierung und extrahieren **ausschließlich** die Eingabe-Embedding-Matrix (30.522 × 768). Der BERT-Encoder wird explizit nicht verwendet. Die Embedding-Matrix wird als feste, nicht-trainierbare Matrix gespeichert.

### Schritt 3: Tokenisierung und Textvektoren
Texte werden ohne Special Tokens tokenisiert und auf konfigurierbare Maximallängen beschnitten (Frage: 48, Support: 96, Antwort: 28 Tokens). Der Mittelwert über alle Token-Embeddings ergibt den 768-dimensionalen Textvektor. Leere Texte werden robust als Nullvektor behandelt.

### Schritt 4: Merkmalskonstruktion
Für jedes Kontext-Antwort-Paar wird ein 3072-dimensionaler Merkmalsvektor gebildet: Konkatenation aus Kontextvektor, Antwortvektor, absoluter Differenz und elementweisem Produkt. Dies liefert dem MLP sowohl die Einzelvektoren als auch direkte Vergleichsinformation.

### Schritt 5: Dataset und Collate-Funktion
Die `SciQDataset`-Klasse permutiert die Antwortoptionen (zufällig im Training, deterministisch in Evaluation) und passt das Label entsprechend an. Die Collate-Funktion übernimmt die Batch-Bildung und erzeugt Tensoren der Form `(B, 4, 3072)` für Features und `(B,)` für Labels.

### Schritt 6: MLP-Modell
Das Modell besteht aus: Linear(3072 → 512) → BatchNorm → ReLU → Dropout(0.3) → Linear(512 → 1). Der Eingabetensor wird von `(B, 4, F)` auf `(B×4, F)` umgeformt und nach dem MLP zu `(B, 4)` Logits zurückgeformt. Keine Softmax vor CrossEntropyLoss.

### Schritt 7: Training
Training mit AdamW, CrossEntropyLoss und dem besten Modell basierend auf Validation Accuracy. Ergänzt durch Learning-Rate-Scheduling (ReduceLROnPlateau) und Early Stopping für effizienteres Training.

### Schritt 8: Evaluation und Analyse
Das beste Modell wird auf dem Testdatensatz evaluiert. Zusätzlich erfolgt eine Konfidenzanalyse mit Visualisierung der Vorhersagesicherheit bei korrekten vs. falschen Antworten.

### Schritt 9: `ask_bot` und Demo-Aufrufe
Die Inferenzfunktion `ask_bot` wendet die identische Pipeline auf einzelne Fragen an, ohne Permutation. Es werden drei automatische Aufrufe mit Testdaten sowie zusätzliche Demonstrationen mit eigenen Fragen durchgeführt.